In [1]:
import os
import glob
import json
import random
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 모델 선택: 'redimnet', 'ecapa_tdnn', 또는 'resnet50'
MODEL_TYPE = 'redimnet'  # Part 1 ReDimNet 파이프라인 설정

# 데이터 경로 (로컬 환경 맞춤)
TRAIN_DIR = '../data/train'
VAL_DIR = '../data/val'

EPOCHS = 10
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 사용 디바이스: {device}")
print(f"✅ 선택된 모델: {MODEL_TYPE}")

# 모델별 최적 설정 자동 할당
if MODEL_TYPE == 'resnet50':
    N_MELS, N_FFT, HOP_LENGTH = 128, 2048, 512
elif MODEL_TYPE in ['redimnet', 'ecapa_tdnn']:
    N_MELS, N_FFT, HOP_LENGTH = 80, 512, 160

print(f"✅ 음향 특징 설정: n_mels={N_MELS}, n_fft={N_FFT}, hop_length={HOP_LENGTH}")


✅ 사용 디바이스: cuda
✅ 선택된 모델: redimnet
✅ 음향 특징 설정: n_mels=80, n_fft=512, hop_length=160


In [2]:
class DCCAudioDataset(Dataset):
    def __init__(self, data_dir, is_train=True, max_files=None, n_mels=128, n_fft=2048, hop_length=512, window_sec=3.0, padding_mode="zero"):
        self.data_dir = data_dir
        self.is_train = is_train
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.window_sec = window_sec
        self.padding_mode = padding_mode
        self.sr = 16000
        self.target_samples = int(self.sr * self.window_sec)
        
        self.samples = self._parse_data(max_files)
        
    def _parse_data(self, max_files):
        samples = []
        label_dir = os.path.join(self.data_dir, 'label')
        audio_dir = os.path.join(self.data_dir, 'audio')
        
        # Fallback 구조
        if not os.path.exists(label_dir):
            label_dir = self.data_dir 
            audio_dir = self.data_dir 
            
        json_files = glob.glob(os.path.join(label_dir, '**', '*.json'), recursive=True)
        if max_files:
            json_files = json_files[:max_files]
            
        for jf in json_files:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 음성 파일 경로 매핑
            wav_name = os.path.basename(jf).replace('.json', '.wav')
            wav_path = os.path.join(audio_dir, wav_name)
            
            if not os.path.exists(wav_path):
                wav_search = glob.glob(os.path.join(audio_dir, '**', wav_name), recursive=True)
                if wav_search:
                    wav_path = wav_search[0]
                else:
                    continue 
                    
            if 'utterances' in data:
                for utt in data['utterances']:
                    start_at = float(utt.get('startAt', 0))
                    end_at = float(utt.get('endAt', 0))
                    speaker = int(utt.get('speaker', 0))
                    
                    samples.append({
                        'wav_path': wav_path,
                        'start_ms': start_at,
                        'end_ms': end_at,
                        'label': speaker
                    })
        return samples
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        wav_path = sample['wav_path']
        
        # 1. 오디오 로드 (특정 구간) - 안전한 librosa 로직
        try:
            start_sec = sample['start_ms'] / 1000.0
            end_sec = sample['end_ms'] / 1000.0
            clip_dur = max(0.01, end_sec - start_sec)
            
            y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=clip_dur)
        except Exception:
            y = np.zeros(self.target_samples, dtype=np.float32)
            
        # 2. 오디오 길이 맞춤 (Padding or Crop)
        cur_len = len(y)
        if cur_len < self.target_samples:
            pad_needed = self.target_samples - cur_len
            if self.padding_mode == "repeat" and cur_len > 0:
                repeats = int(np.ceil(self.target_samples / cur_len))
                y = np.tile(y, repeats)[:self.target_samples]
            else: # zero padding
                y = np.pad(y, (0, pad_needed), mode='constant')
        elif cur_len > self.target_samples:
            if self.is_train:
                max_start = cur_len - self.target_samples
                start_idx = random.randint(0, max_start)
            else:
                start_idx = (cur_len - self.target_samples) // 2
            y = y[start_idx : start_idx + self.target_samples]

        # 3. Safe STFT 방어: 길이가 n_fft보다 작으면 추가 패딩
        if len(y) < self.n_fft:
            y = np.pad(y, (0, self.n_fft - len(y)), mode='constant')

        # 4. Mel-Spectrogram 변환 (dB scale)
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_fft=self.n_fft,
            hop_length=self.hop_length, n_mels=self.n_mels
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)

        # 5. 정규화 (Min-Max) & 1채널 텐서 변환
        mel_norm = (mel_db + 80.0) / 80.0
        mel_norm = np.clip(mel_norm, 0.0, 1.0)
        
        tensor_x = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0) # (1, n_mels, time)
        tensor_y = torch.tensor(sample['label'], dtype=torch.float32)
            
        return tensor_x, tensor_y


In [3]:
def get_audio_resnet50(dropout_rate=0.3):
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    
    # 3채널 -> 1채널 가중치 평균 변환
    old_conv = model.conv1
    new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                         stride=old_conv.stride, padding=old_conv.padding, bias=False)
    new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
    model.conv1 = new_conv
    
    # FC 레이어 수정 (1 클래스 - BCEWithLogitsLoss 용)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 1)
    )
    return model

from benchmark_suite.models.redimnet import ReDimNet2_B2
from benchmark_suite.models.ecapa_tdnn import ECAPA_TDNN

def build_selected_model(model_type):
    if model_type == 'resnet50':
        return get_audio_resnet50()
    elif model_type == 'redimnet':
        # ReDimNet도 1개의 logit을 출력하도록 설정 (BCEWithLogitsLoss 용)
        return ReDimNet2_B2(num_classes=1)
    elif model_type == 'ecapa_tdnn':
        return ECAPA_TDNN(in_channels=80, channels=512, num_classes=1)
    else:
        raise ValueError("Invalid model_type. Choose 'resnet50', 'redimnet', or 'ecapa_tdnn'.")


In [4]:
# ==========================================
# 🧪 Sanity Check (데이터 로더 & 모델 순전파/역전파 검증)
# ==========================================
print("--- [Sanity Check 시작] ---")

# 1. 소규모 데이터로 데이터로더 생성 (최대 10개 파일만 파싱)
sanity_dataset = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=10, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
if len(sanity_dataset) == 0:
    print(f"⚠️ 경고: '{TRAIN_DIR}' 에서 샘플을 찾지 못했습니다. 경로에 데이터가 있는지 확인하세요.")
else:
    sanity_loader = DataLoader(sanity_dataset, batch_size=2, shuffle=True)
    
    # 2. 모델 및 옵티마이저 생성
    sanity_model = build_selected_model(MODEL_TYPE).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(sanity_model.parameters(), lr=1e-4)
    scaler = GradScaler()
    
    sanity_model.train()
    
    try:
        inputs, labels = next(iter(sanity_loader))
        inputs = inputs.to(device)
        labels = labels.to(device).unsqueeze(1)
        
        print(f"입력 텐서 모양: {inputs.shape}")
        print(f"라벨 텐서 모양: {labels.shape}")
        
        optimizer.zero_grad()
        with autocast():
            outputs = sanity_model(inputs)
            loss = criterion(outputs, labels)
            
        print(f"모델 출력 모양: {outputs.shape}")
        print(f"손실값 (Loss): {loss.item():.4f}")
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if torch.isnan(loss):
            print("❌ 오류: Loss가 NaN입니다! 입력 데이터나 정규화를 확인하세요.")
        else:
            print("✅ Forward/Backward 패스 성공! 모델 학습이 정상적으로 가능합니다.")
            
    except Exception as e:
        print(f"❌ Sanity Check 실패: {e}")
        
print("--- [Sanity Check 종료] ---")


--- [Sanity Check 시작] ---
입력 텐서 모양: torch.Size([2, 1, 80, 301])
라벨 텐서 모양: torch.Size([2, 1])
모델 출력 모양: torch.Size([2, 1])
손실값 (Loss): 0.5325
✅ Forward/Backward 패스 성공! 모델 학습이 정상적으로 가능합니다.
--- [Sanity Check 종료] ---


In [5]:
def train_model():
    print("="*60)
    print(f"🚀 [RTX 3060 전용] {MODEL_TYPE} Full Training Pipeline (90%+ 버전)")
    print(f" - Train Dir: {TRAIN_DIR}")
    print(f" - Val Dir: {VAL_DIR}")
    print(f" - Batch Size: {BATCH_SIZE} | Epochs: {EPOCHS}")
    print(f" - AMP 활성화 | BCEWithLogitsLoss + Random Crop")
    print("="*60)
    
    if not os.path.exists(TRAIN_DIR):
        print(f"⚠️ 에러: 학습 데이터 폴더 '{TRAIN_DIR}'를 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    train_dataset = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    val_dataset = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    print(f"✅ 데이터 로드 완료! (Train: {len(train_dataset):,} 샘플 | Val: {len(val_dataset):,} 샘플)")

    model = build_selected_model(MODEL_TYPE).to(device)

    # ---------------------------------------------------------
    # 🔄 [핵심] 이전 벤치마크/학습에서 저장된 가중치 불러오기
    # ---------------------------------------------------------
    checkpoint_path = f"test_results/checkpoints/best_{MODEL_TYPE}.pt"
    if os.path.exists(checkpoint_path):
        print(f"📦 저장된 가중치 발견! 불러옵니다: {checkpoint_path}")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            else:
                state_dict = checkpoint
            
            # 크기가 일치하는 가중치만 안전하게 필터링하여 덮어쓰기 (Shape mismatch 원천 차단!)
            model_dict = model.state_dict()
            pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)
            
            print("✅ 가중치 로드 성공! (분류기를 제외한 핵심 특징 추출기 완벽 복구 완료)")
        except Exception as e:
            print(f"⚠️ 가중치 로드 실패: {e}")
    else:
        print(f"🌱 저장된 가중치가 없습니다 ({checkpoint_path}). 바닥부터 새로 학습합니다.")
    # ---------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    scaler = GradScaler()
    best_acc = 0.0
    os.makedirs('checkpoints', exist_ok=True)
    
    for epoch in range(1, EPOCHS + 1):
        # --- Training ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * inputs.size(0)
            
            # 예측값 계산 (Sigmoid >= 0.5)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            train_total += labels.size(0)
            train_correct += preds.eq(labels).sum().item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        scheduler.step()
        train_acc = train_correct / train_total
        train_loss = train_loss / train_total
        
        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
                
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()
                
        val_acc = val_correct / val_total
        val_loss = val_loss / val_total
        
        print(f"📋 Epoch {epoch:02d} 결과 | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Val Acc: {val_acc*100:.2f}%")
        
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = f"checkpoints/best_{MODEL_TYPE}.pt"
            torch.save(model.state_dict(), save_path)
            print(f"⭐ [신기록 달성!] 가중치 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)")


In [6]:
# 전체 데이터 풀 학습 시작
# 주의: 시간이 오래 걸릴 수 있으므로 Sanity Check가 통과된 이후에만 실행하세요.
train_model()


🚀 [RTX 3060 전용] redimnet Full Training Pipeline (90%+ 버전)
 - Train Dir: ../data/train
 - Val Dir: ../data/val
 - Batch Size: 32 | Epochs: 10
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
✅ 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
📦 저장된 가중치 발견! 불러옵니다: test_results/checkpoints/best_redimnet.pt
✅ 가중치 로드 성공! (분류기를 제외한 핵심 특징 추출기 완벽 복구 완료)


Epoch 1/10 [Val]: 100%|██████████| 3498/3498 [11:36<00:00,  5.02it/s]


📋 Epoch 01 결과 | Train Acc: 87.83% | Val Loss: 0.2123 Val Acc: 90.01%
⭐ [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 90.01%)


Epoch 2/10 [Val]: 100%|██████████| 3498/3498 [11:11<00:00,  5.21it/s]


📋 Epoch 02 결과 | Train Acc: 89.95% | Val Loss: 0.2025 Val Acc: 90.59%
⭐ [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 90.59%)


...

📋 Epoch 08 결과 | Train Acc: 92.73% | Val Loss: 0.1755 Val Acc: 91.75%
⭐ [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 91.75%)


Epoch 9/10 [Val]: 100%|██████████| 3498/3498 [10:39<00:00,  5.47it/s]


📋 Epoch 09 결과 | Train Acc: 93.07% | Val Loss: 0.1737 Val Acc: 91.89%
⭐ [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 91.89%)


Epoch 10/10 [Val]: 100%|██████████| 3498/3498 [10:51<00:00,  5.37it/s]

📋 Epoch 10 결과 | Train Acc: 93.26% | Val Loss: 0.1783 Val Acc: 91.80%


# ============================================================
# ⚡ [Part 2] ECAPA-TDNN 풀학습 파이프라인 (화자 음색 시계열 모델)
# ============================================================
> **특징**: 1D 시간 지연 합성곱(TDNN) + Squeeze-and-Excitation + 통계적 풀링 구조  
> **목적**: ReDimNet(2D Conv + MHA)과 완전히 다른 시계열 음색 특징을 학습하여 앙상블 시너지 극대화  
> **체크포인트 저장 위치**: `checkpoints/best_ecapa_tdnn.pt`

In [7]:
# ============================================================
# 🚀 ECAPA-TDNN 독립 풀학습 실행 셀
# (ReDimNet의 이전 학습 로그를 건드리지 않고 독립적으로 실행됩니다)
# ============================================================
def train_ecapa_model():
    ecapa_type = 'ecapa_tdnn'
    ecapa_n_mels, ecapa_n_fft, ecapa_hop = 80, 512, 160
    ecapa_epochs = 10
    ecapa_batch_size = 32
    
    print('=' * 65)
    print(f'🚀 [RTX 3060 전용] {ecapa_type.upper()} Full Training Pipeline 시작')
    print(f' - Train: {TRAIN_DIR} | Val: {VAL_DIR}')
    print(f' - Batch: {ecapa_batch_size} | Epochs: {ecapa_epochs} | n_mels: {ecapa_n_mels}')
    print(f' - AMP 활성화 | BCEWithLogitsLoss + Random Crop')
    print('=' * 65)
    
    # 1. 데이터셋 & 데이터로더 생성
    train_ds = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=ecapa_n_mels, n_fft=ecapa_n_fft, hop_length=ecapa_hop)
    val_ds = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=ecapa_n_mels, n_fft=ecapa_n_fft, hop_length=ecapa_hop)
    
    train_loader = DataLoader(train_ds, batch_size=ecapa_batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=ecapa_batch_size, shuffle=False, num_workers=0, pin_memory=True)
    print(f'✅ 데이터 로드 완료! (Train: {len(train_ds):,} 샘플 | Val: {len(val_ds):,} 샘플)')
    
    # 2. ECAPA-TDNN 모델 생성
    model = build_selected_model('ecapa_tdnn').to(device)
    
    # 3. 벤치마크 사전 가중치 불러오기 (전이학습)
    ckpt_path = 'test_results/checkpoints/best_ecapa_tdnn.pt'
    if os.path.exists(ckpt_path):
        print(f'📦 저장된 가중치 발견! 불러옵니다: {ckpt_path}')
        try:
            ckpt = torch.load(ckpt_path, map_location=device)
            state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
            m_dict = model.state_dict()
            matched = {k: v for k, v in state_dict.items() if k in m_dict and v.shape == m_dict[k].shape}
            m_dict.update(matched)
            model.load_state_dict(m_dict)
            print(f'✅ 가중치 로드 성공! (일치 레이어 {len(matched)}개 복구 완료)')
        except Exception as e:
            print(f'⚠️ 가중치 로드 실패: {e}')
    else:
        print('🌱 이전 가중치가 없어 바닥부터 새로 학습합니다.')
        
    # 4. 손실함수, 옵티마이저, 스케줄러
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=ecapa_epochs, eta_min=1e-6)
    scaler = GradScaler()
    best_acc = 0.0
    
    # 5. 에폭 루프
    for epoch in range(1, ecapa_epochs + 1):
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{ecapa_epochs} [Train ECAPA]')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            tr_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            tr_total += labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        scheduler.step()
        tr_acc = tr_correct / tr_total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{ecapa_epochs} [Val ECAPA]'):
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()
                
        val_acc = val_correct / val_total
        val_loss = val_loss / val_total
        
        print(f'📋 [ECAPA-TDNN] Epoch {epoch:02d} | Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = 'checkpoints/best_ecapa_tdnn.pt'
            torch.save(model.state_dict(), save_path)
            print(f'⭐ [ECAPA 신기록!] 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)')

# ECAPA-TDNN 풀학습 실행
train_ecapa_model()


🚀 [RTX 3060 전용] ECAPA_TDNN Full Training Pipeline 시작
 - Train: ../data/train | Val: ../data/val
 - Batch: 32 | Epochs: 10 | n_mels: 80
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
✅ 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
📦 저장된 가중치 발견! 불러옵니다: test_results/checkpoints/best_ecapa_tdnn.pt
✅ 가중치 로드 성공! (일치 레이어 231개 복구 완료)


Epoch 1/10 [Val ECAPA]: 100%|██████████| 3498/3498 [12:06<00:00,  4.82it/s]


📋 [ECAPA-TDNN] Epoch 01 | Train Acc: 87.97% | Val Loss: 0.2160 | Val Acc: 90.01%
⭐ [ECAPA 신기록!] 저장 완료: checkpoints/best_ecapa_tdnn.pt (Acc: 90.01%)


Epoch 2/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:22<00:00,  5.12it/s]


📋 [ECAPA-TDNN] Epoch 02 | Train Acc: 90.12% | Val Loss: 0.2192 | Val Acc: 89.59%


...

📋 [ECAPA-TDNN] Epoch 08 | Train Acc: 93.07% | Val Loss: 0.1706 | Val Acc: 92.10%
⭐ [ECAPA 신기록!] 저장 완료: checkpoints/best_ecapa_tdnn.pt (Acc: 92.10%)


Epoch 9/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:15<00:00,  5.18it/s]


📋 [ECAPA-TDNN] Epoch 09 | Train Acc: 93.40% | Val Loss: 0.1724 | Val Acc: 92.09%


Epoch 10/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:59<00:00,  4.86it/s]

📋 [ECAPA-TDNN] Epoch 10 | Train Acc: 93.69% | Val Loss: 0.1764 | Val Acc: 92.08%


# ============================================================
# 🏢 [Part 2.5] AudioResNet-50 풀학습 파이프라인 (2D 비전 텍스처 앵커)
# ============================================================
> **특징**: 2D ImageNet 사전학습 가중치 기반 전이학습 (23.50M 파라미터)  
> **음향 설정**: `n_mels=128`, `n_fft=2048`, `hop_length=512` (고해상도 2D 멜 스펙트로그램)  
> **목적**: 1차 90.20% 베이스라인을 풀데이터(87만 샘플)로 재학습하여 91%+ 로 도약 및 앙상블 앵커 확보  
> **체크포인트 저장 위치**: `checkpoints/best_resnet50.pt`

In [8]:
# ============================================================
# 🚀 AudioResNet-50 독립 풀학습 실행 셀
# (ReDimNet과 ECAPA의 학습 로그를 전혀 건드리지 않고 독립적으로 실행됩니다)
# ============================================================
def train_resnet_model():
    resnet_type = 'resnet50'
    resnet_n_mels, resnet_n_fft, resnet_hop = 128, 2048, 512
    resnet_epochs = 10
    resnet_batch_size = 32
    
    print('=' * 65)
    print(f'🚀 [RTX 3060 전용] AudioResNet-50 Full Training Pipeline 시작')
    print(f' - Train: {TRAIN_DIR} | Val: {VAL_DIR}')
    print(f' - Batch: {resnet_batch_size} | Epochs: {resnet_epochs} | n_mels: {resnet_n_mels}')
    print(f' - AMP 활성화 | BCEWithLogitsLoss + Random Crop')
    print('=' * 65)
    
    # 1. 데이터셋 & 데이터로더 생성 (ResNet 전용 n_mels=128 고해상도)
    train_ds = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=resnet_n_mels, n_fft=resnet_n_fft, hop_length=resnet_hop)
    val_ds = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=resnet_n_mels, n_fft=resnet_n_fft, hop_length=resnet_hop)
    
    train_loader = DataLoader(train_ds, batch_size=resnet_batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=resnet_batch_size, shuffle=False, num_workers=0, pin_memory=True)
    print(f'✅ 데이터 로드 완료! (Train: {len(train_ds):,} 샘플 | Val: {len(val_ds):,} 샘플)')
    
    # 2. ResNet-50 모델 생성
    model = build_selected_model('resnet50').to(device)
    
    # 3. 기존 90% 베이스라인 가중치 불러오기 (전이학습)
    ckpt_candidates = [
        'results_benchmark/checkpoints/baseline_resnet50_90.pt',
        'test_results/checkpoints/best_resnet50.pt',
        'results/checkpoints/best_resnet50.pt'
    ]
    loaded = False
    for ckpt_path in ckpt_candidates:
        if os.path.exists(ckpt_path):
            print(f'📦 저장된 ResNet 가중치 발견! 불러옵니다: {ckpt_path}')
            try:
                ckpt = torch.load(ckpt_path, map_location=device)
                state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
                m_dict = model.state_dict()
                matched = {k: v for k, v in state_dict.items() if k in m_dict and v.shape == m_dict[k].shape}
                m_dict.update(matched)
                model.load_state_dict(m_dict)
                print(f'✅ ResNet 가중치 로드 성공! (일치 레이어 {len(matched)}개 복구 완료)')
                loaded = True
                break
            except Exception as e:
                print(f'⚠️ 가중치 로드 실패: {e}')
    if not loaded:
        print('🌱 이전 가중치가 없어 ImageNet 사전학습 가중치 기반으로 학습합니다.')
        
    # 4. 손실함수, 옵티마이저, 스케줄러
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=resnet_epochs, eta_min=1e-6)
    scaler = GradScaler()
    best_acc = 0.0
    
    # 5. 에폭 루프
    for epoch in range(1, resnet_epochs + 1):
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{resnet_epochs} [Train ResNet]')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            tr_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            tr_total += labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        scheduler.step()
        tr_acc = tr_correct / tr_total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{resnet_epochs} [Val ResNet]'):
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()
                
        val_acc = val_correct / val_total
        val_loss = val_loss / val_total
        
        print(f'📋 [ResNet-50] Epoch {epoch:02d} | Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
        if val_acc > best_acc:
            best_acc = val_acc
            save_path = 'checkpoints/best_resnet50.pt'
            torch.save(model.state_dict(), save_path)
            print(f'⭐ [ResNet 신기록!] 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)')

# AudioResNet-50 풀학습 실행
train_resnet_model()


🚀 [RTX 3060 전용] AudioResNet-50 Full Training Pipeline 시작
 - Train: ../data/train | Val: ../data/val
 - Batch: 32 | Epochs: 10 | n_mels: 128
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
✅ 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
📦 저장된 ResNet 가중치 발견! 불러옵니다: results_benchmark/checkpoints/baseline_resnet50_90.pt
✅ ResNet 가중치 로드 성공! (일치 레이어 0개 복구 완료)


Epoch 1/10 [Val ResNet]: 100%|██████████| 3498/3498 [14:02<00:00,  4.15it/s]


📋 [ResNet-50] Epoch 01 | Train Acc: 88.56% | Val Loss: 0.2159 | Val Acc: 89.90%
⭐ [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 89.90%)


Epoch 2/10 [Val ResNet]: 100%|██████████| 3498/3498 [13:22<00:00,  4.36it/s]


📋 [ResNet-50] Epoch 02 | Train Acc: 90.75% | Val Loss: 0.2004 | Val Acc: 90.61%
⭐ [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 90.61%)


...

📋 [ResNet-50] Epoch 08 | Train Acc: 94.80% | Val Loss: 0.2737 | Val Acc: 91.14%


Epoch 9/10 [Val ResNet]: 100%|██████████| 3498/3498 [12:21<00:00,  4.72it/s]


📋 [ResNet-50] Epoch 09 | Train Acc: 95.02% | Val Loss: 0.3271 | Val Acc: 91.19%
⭐ [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 91.19%)


Epoch 10/10 [Val ResNet]: 100%|██████████| 3498/3498 [12:46<00:00,  4.56it/s]


📋 [ResNet-50] Epoch 10 | Train Acc: 95.12% | Val Loss: 0.3656 | Val Acc: 91.20%
⭐ [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 91.20%)


# ============================================================
# 📊 [Part 3] 모델별 다차원 효율성 및 정확도 비교 벤치마크
# ============================================================
> **평가 항목**: 정확도(Val Acc)뿐만 아니라 **단일 지연시간(Latency, ms)**, **실시간성 지수(RTF)**, **초당 처리량(Throughput)**, **가중치 파일 크기(MB)**, **파라미터 수(M)**를 실측 비교합니다.

In [ ]:
import time
import pandas as pd

print('📊 [효율성 정밀 실측 벤치마크 시작 (RTX 3060 기준)]')

benchmark_targets = [
    ('ReDimNet2-B2', ReDimNet2_B2(num_classes=1), (1, 80, 300), 'checkpoints/best_redimnet.pt', 91.89),
    ('ECAPA-TDNN', ECAPA_TDNN(in_channels=80, channels=512, num_classes=1), (1, 80, 300), 'checkpoints/best_ecapa_tdnn.pt', 92.10),
    ('AudioResNet-50', get_audio_resnet50(), (1, 1, 128, 300), 'checkpoints/best_resnet50.pt', 91.20)
]

bench_rows = []
for name, model, in_shape, ckpt_p, known_acc in benchmark_targets:
    model = model.to(device)
    model.eval()
    
    # 1. 파라미터 & 파일 크기
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    size_mb = (os.path.getsize(ckpt_p) / (1024*1024)) if os.path.exists(ckpt_p) else 0.0
    
    # 2. Warmup & Latency 측정 (200회 반복)
    dummy = torch.randn(*in_shape, device=device)
    with torch.no_grad():
        for _ in range(30):
            _ = model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
        
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(200):
            _ = model(dummy)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    t1 = time.perf_counter()
    lat_ms = ((t1 - t0) / 200) * 1000
    rtf = (lat_ms / 1000.0) / 3.0  # 3초 발화 기준 RTF
    
    # 3. Throughput 측정 (batch=32, 50회)
    batch_shape = (32,) + in_shape[1:]
    dummy_b = torch.randn(*batch_shape, device=device)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(50):
            _ = model(dummy_b)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    t1 = time.perf_counter()
    throughput = (32 * 50) / (t1 - t0)
    
    bench_rows.append({
        '모델명': name,
        '파라미터': f'{params_m:.2f} M',
        '파일크기': f'{size_mb:.1f} MB',
        '지연시간': f'{lat_ms:.2f} ms',
        'RTF(실시간성)': f'{rtf:.4f}',
        '초당 처리량': f'{throughput:.1f} 건/초',
        '최고 정확도': f'{known_acc:.2f}%' if known_acc else '학습 후 반영'
    })

df_bench = pd.DataFrame(bench_rows)
print('\n' + '=' * 80)
print('📊 [RTX 3060 실측] 모델별 다차원 효율성 비교표')
print('=' * 80)
display(df_bench)


# ============================================================
# 🤝 [Part 4] 3대장 Soft Voting 앙상블 & Threshold Tuning (최종 93~95%+ 도전)
# ============================================================
> **조합**: ResNet-50 (2D 시각 텍스처) + ReDimNet (하이브리드 MHA) + ECAPA-TDNN (화자 음색 시계열)  
> **원리**: 각 모델의 예측 확률(Probability)을 가중 평균하고, F1-Score가 최대가 되는 최적 컷오프(임계값)를 그리드 서치합니다.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print('🚀 [3대장 Soft Voting 앙상블 검증 시작]')

# 가중치 파일 존재 확인
w_resnet = 'checkpoints/best_resnet50.pt' if os.path.exists('checkpoints/best_resnet50.pt') else 'results_benchmark/checkpoints/baseline_resnet50_90.pt'
w_redim = 'checkpoints/best_redimnet.pt'
w_ecapa = 'checkpoints/best_ecapa_tdnn.pt' if os.path.exists('checkpoints/best_ecapa_tdnn.pt') else 'test_results/checkpoints/best_ecapa_tdnn.pt'

print(f'1. ResNet-50: {w_resnet} (존재: {os.path.exists(w_resnet)})')
print(f'2. ReDimNet:   {w_redim} (존재: {os.path.exists(w_redim)})')
print(f'3. ECAPA-TDNN: {w_ecapa} (존재: {os.path.exists(w_ecapa)})')

# 검증 데이터셋 로드 (평가용 5,000개 샘플 빠른 평가)
eval_ds_80 = DCCAudioDataset(VAL_DIR, is_train=False, max_files=50, n_mels=80, n_fft=512, hop_length=160)
eval_ds_128 = DCCAudioDataset(VAL_DIR, is_train=False, max_files=50, n_mels=128, n_fft=2048, hop_length=512)
print(f'✅ 앙상블 평가용 검증 샘플 로드 완료: {len(eval_ds_80):,}개')
